[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *Polars*.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np

## ¿Por qué *Polars*?

*Polars* es un motor de análisis de datos moderno, escrito en *Rust* y diseñado para ser extremadamente rápido y eficiente en memoria. A diferencia de *Pandas*, que puede ser lento con *datasets* grandes, *Polars* está optimizado para:

* **Velocidad:** implementación en *Rust* con evaluación *lazy*.
* **Memoria:** uso eficiente de *RAM* con *Arrow* como *backend*.
* **API intuitiva:** expresiones composables similares a *SQL*.
* **Escalabilidad:** diseñado para *Big Data*.

### Comparativa: *Pandas* vs *Polars*

| Aspecto | Pandas | Polars |
| :--- | :--- | :--- |
| **Lenguaje** | Python | Rust + Python binding |
| **Backend** | NumPy | Arrow |
| **Velocidad** | Moderada | ⚡ Muy rápida |
| **Memoria** | Eficiente | 🎯 Óptima |
| **Lazy eval.** | No | Sí (opcional) |
| **API** | Imperativos | Expresiones |
| **Escalabilidad** | En-memoria | In-memory + distribuido |

### Benchmarks Reales

En operaciones típicas de análisis de datos, *Polars* puede ser **5-10x más rápido** que *Pandas*.

* La siguiente celda mide el tiempo de creación de un `DataFrame` con 1 millón de filas en *Pandas* y *Polars*, para ilustrar la diferencia de rendimiento entre ambas bibliotecas.

In [ ]:
# Benchmark: Creación de DataFrame
import time

n_rows = 1_000_000

# Benchmark Pandas
start = time.time()
df_pandas = pd.DataFrame({
    'id': range(n_rows),
    'valor': np.random.rand(n_rows)
})
tiempo_pandas = time.time() - start

# Benchmark Polars
start = time.time()
df_polars = pl.DataFrame({
    'id': range(n_rows),
    'valor': np.random.rand(n_rows)
})
tiempo_polars = time.time() - start

print(f"Pandas: {tiempo_pandas:.4f}s")
print(f"Polars: {tiempo_polars:.4f}s")
print(f"Polars es {tiempo_pandas/tiempo_polars:.1f}x más rápido")

## *DataFrames* en *Polars*.

Un `DataFrame` en *Polars* es similar al de *Pandas*, pero con una *API* más eficiente y expresiva.

* La siguiente celda crea un `DataFrame` de ejemplo con columnas de tipo cadena, entero y flotante.

In [ ]:
df = pl.DataFrame({
    'nombre':   ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'edad':     [25, 30, 35, 28, 32],
    'salario':  [50000.0, 60000.0, 75000.0, 55000.0, 70000.0],
})
df

* La siguiente celda inspecciona el `DataFrame`: forma (`shape`), nombres de columnas (`columns`), schema tipado y lista de tipos de datos (`dtypes`).

In [ ]:
# Inspeccionar el DataFrame
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns}")
print(f"\nSchema:\n{df.schema}")
print(f"\nDtypes:\n{df.dtypes}")

* La siguiente celda muestra las primeras y últimas filas del `DataFrame` con `.head()` y `.tail()`.

In [ ]:
# Head y tail
print("Head:")
print(df.head(2))
print("\nTail:")
print(df.tail(2))

* La siguiente celda llama a `df.describe()` para obtener estadísticas descriptivas (media, desviación estándar, cuartiles) del `DataFrame`.

In [ ]:
# Describe (estadísticas)
print(df.describe())

* La siguiente celda lee un archivo *CSV* con `pl.read_csv()` y devuelve directamente un `DataFrame` de *Polars*, sin pasar por *Pandas*.

In [ ]:
# Leer CSV con Polars
df_csv = pl.read_csv('datos_ejemplo.csv')
print(f"DataFrame desde CSV:\n{df_csv}")

* La siguiente celda lee un archivo *Parquet* con `pl.read_parquet()`. *Parquet* es el formato columnar estándar recomendado para almacenamiento de alta performance.

In [ ]:
# Leer Parquet
df_parquet = pl.read_parquet('tabla_ejemplo.parquet')
print(f"DataFrame desde Parquet:\n{df_parquet}")

## Selecciones y Filtros con Expresiones.

*Polars* utiliza un sistema de **expresiones** que permite operaciones claras y composables.

* La siguiente celda selecciona columnas específicas del `DataFrame` con `.select()`.

In [ ]:
# Select: elegir columnas
df_select = df.select(['nombre', 'edad'])
print("Select columns:")
print(df_select)

* La siguiente celda filtra filas con `.filter()` usando una expresión de *Polars*.

In [ ]:
# Filter: filtrar filas
df_filtered = df.filter(pl.col('edad') > 28)
print("Personas con edad > 28:")
print(df_filtered)

* La siguiente celda aplica un filtro con condición compuesta (operador `&`) para combinar múltiples predicados sobre distintas columnas.

In [ ]:
# Expresiones combinadas
df_complex = df.filter(
    (pl.col('edad') > 25) & (pl.col('salario') >= 60000)
)
print("Edad > 25 AND Salario >= 60000:")
print(df_complex)

## Transformaciones.

Agregar, modificar o eliminar columnas con `with_columns()`.

* La siguiente celda agrega columnas calculadas al `DataFrame` con `.with_columns()`, usando `.alias()` para asignar nombres.

In [ ]:
# Agregar columnas
df_transformed = df.with_columns(
    (pl.col('salario') * 1.1).alias('salario_incrementado'),
    (pl.col('edad') / 10).alias('edad_decada')
)
print("Con columnas calculadas:")
print(df_transformed)

* La siguiente celda renombra columnas con `.rename()`, que recibe un diccionario `{nombre_original: nuevo_nombre}`.

In [ ]:
# Renombrar columnas
df_renamed = df.rename({
    'nombre': 'employee_name',
    'edad': 'employee_age'
})
print(df_renamed)

* La siguiente celda elimina una columna con `.drop()`, que acepta un nombre o una lista de nombres de columnas.

In [ ]:
# Drop (eliminar) columnas
df_dropped = df.drop('salario')
print('Sin columna salario:')
print(df_dropped)

## *Groupby* y Agregaciones.

Agrupar datos y aplicar funciones de agregación de forma eficiente.

* La siguiente celda crea `df_sales`, un `DataFrame` con datos de empleados por departamento que se usará en los ejemplos de agregación.

In [ ]:
df_sales = pl.DataFrame({
    'empleado':     ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'departamento': ['Ventas', 'IT', 'Ventas', 'IT', 'RRHH', 'RRHH'],
    'salario':      [55000.0, 70000.0, 60000.0, 75000.0, 50000.0, 52000.0],
})
df_sales

* La siguiente celda agrupa por `departamento` con `.group_by()` y aplica múltiples funciones de agregación con `.agg()`.

In [ ]:
# Groupby con agregaciones
df_agg = df_sales.group_by('departamento').agg(
    pl.col('salario').mean().alias('salario_promedio'),
    pl.col('salario').max().alias('salario_max'),
    pl.col('empleado').count().alias('cantidad')
)
print("Aggregations por departamento:")
print(df_agg)

* La siguiente celda ordena los resultados por `salario` de forma descendente con `.sort()`.

In [ ]:
# Ordenar resultados
df_sorted = df_sales.sort('salario', descending=True)
print("Ordenado por salario (descendente):")
print(df_sorted)

## *Lazy Evaluation*.

*Polars* soporta ***evaluación lazy***, donde las operaciones se definen pero no se ejecutan hasta que se llama `.collect()`.

Esto permite que *Polars* optimice automáticamente las consultas.

* La siguiente celda construye una *query lazy* con `.lazy()` y la ejecuta con `.collect()`. Hasta que se llama `collect()`, *Polars* solo construye el plan de ejecución sin materializar datos.

In [ ]:
# Evaluación lazy
query = (df
    .lazy()
    .filter(pl.col('edad') > 27)
    .select(['nombre', 'salario'])
    .sort('salario', descending=True)
)

print(f"Query type: {type(query)}")
print("\nEjecutar query:")
result = query.collect()
print(result)

* La siguiente celda imprime el plan de ejecución optimizado de la *query* con `.explain()`, mostrando cómo *Polars* reorganiza y optimiza internamente las operaciones (*predicate push-down*, eliminación de columnas no usadas).

In [ ]:
# Ver el plan de ejecución optimizado
query_plan = (df
    .lazy()
    .filter(pl.col('edad') > 27)
    .select(['nombre', 'salario'])
    .sort('salario')
)

print("Query plan (optimizado):")
print(query_plan.explain())

## Conversiones *Pandas* ↔ *Polars*.

Es fácil convertir entre *Pandas* y *Polars*.

* La siguiente celda convierte un *DataFrame* de *Pandas* a uno de *Polars* con `pl.from_pandas()`.

In [ ]:
# Pandas → Polars
df_pandas = pd.DataFrame({
    'A': [1, 2, 3],
    'B': ['x', 'y', 'z']
})
df_polars = pl.from_pandas(df_pandas)
print(f"Pandas a Polars:\n{df_polars}")

* La siguiente celda convierte un *DataFrame* de *Polars* a uno de *Pandas* con `.to_pandas()`, verificando el tipo del objeto resultante.

In [ ]:
# Polars → Pandas
df_back_to_pandas = df_polars.to_pandas()
print(f"Polars a Pandas:\n{df_back_to_pandas}")
print(f"\nTipo: {type(df_back_to_pandas)}")

## Resumen.

*Polars* ofrece un motor de análisis de datos moderno con ventajas claras sobre *Pandas* en escenarios de alto rendimiento:

| Concepto | Descripción |
| :--- | :--- |
| **`DataFrame`** | Estructura tabular columnar, similar a *Pandas* pero con *API* de expresiones |
| **Expresiones** | Sistema composable para seleccionar, filtrar y transformar datos |
| **`group_by`** | Agrupación y agregación eficiente sobre columnas categóricas |
| ***Lazy evaluation*** | Construcción de planes de consulta optimizados antes de ejecutar |
| **Conversiones** | Interoperabilidad directa con *Pandas* mediante `from_pandas()` y `to_pandas()` |

En los próximos notebooks se verán **expresiones avanzadas** de *Polars*: *window functions*, *joins* optimizados y *lazy evaluation* profunda.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>